In [ ]:
import os
from glob import glob
from enum import StrEnum, auto
from functools import partial

from natsort import natsorted
import numpy as np
import torch
from matplotlib import pyplot as plt
from tifffile import imread
from torchvision.transforms import v2
from torchvision.tv_tensors import Image, Mask

from segmentation_utils import scale_intensities


class NormalizationStrategy(StrEnum):
    PER_IMAGE = auto(),
    PER_PLANE = auto(),
    

# TODO: sparse labelling does not really play a role for this class, rename?
class SparseLabeledImageDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        img_files,
        mask_files,
        transforms=None,
        plane_selectors=None,
        normalization_strategy: NormalizationStrategy=None,
        normalization_quantiles=(0.0, 1.0),
        plane_sliding_window=1
    ):
        """
        PyTorch Dataset of sparsely labelled (0 considered unlabelled) TIF stacks.
        Will only include planes with a minimum number of nonzero pixels.
        
        Parameters
        ----------
        img_files: list of [str/Path]
            input (intensity) image file paths
        mask_files: list of [str/Path]
            mask / label file paths, must match img_files
        transforms: torchvision v2 transform
            transforms to be applied to images. should only be for augmentation
            conversion to tensor is done already
        plane_selectors: list/iterable of callables
            callables that return a selection along the first dimension when applied to a 3D mask
        """

        self.images = []
        self.masks = []
        self.transforms = transforms

        for img_file, mask_file in zip(img_files, mask_files):

            img = imread(img_file)
            mask = imread(mask_file)

            # add dummy z axis for 2D data
            if img.ndim == 2:
                img = img[np.newaxis]
                mask = mask[np.newaxis]

            # normalize intensities (or not)
            # NOTE: do it before conversion to torch as torch.quantile seems to struggle with large arrays
            img = SparseLabeledImageDataset._normalize_intensities(img, normalization_strategy, normalization_quantiles)

            if plane_sliding_window > 1:
                img = sliding_window_planewise_padded(img, plane_sliding_window)

            # apply plane selectors
            # those should be callables that return a selection when applied to the mask
            img_selected, mask_selected = img, mask
            if plane_selectors is not None:
                for selector in plane_selectors:
                    selection = selector(mask_selected)
                    img_selected, mask_selected = img_selected[selection], mask_selected[selection]

            # to torch tensors with standard datatypes
            self.images.extend(torch.from_numpy(img_selected).float())
            self.masks.extend(torch.from_numpy(mask_selected).long())

        print(self.images[0].shape)
        # convert to torchvision TVTensors (so augmentations can be easily applied to both img and mask)
        self.images = [Image(img) for img in self.images]
        self.masks = [Mask(mask) for mask in self.masks]

    @staticmethod
    def _normalize_intensities(img, normalization_strategy=None, normalization_quantiles= (0.0, 1.0)):
        
        if normalization_strategy is None:
            return img
        
        elif normalization_strategy == NormalizationStrategy.PER_IMAGE:
            vl, vh = np.quantile(img, normalization_quantiles)
            return scale_intensities(img.astype(np.float32), (vl, vh))
        
        elif normalization_strategy == NormalizationStrategy.PER_PLANE:
            planes = []
            for plane in img:
                vl, vh = np.quantile(plane, normalization_quantiles)
                planes.append( scale_intensities(plane.astype(np.float32), (vl, vh)) )
            return np.stack(planes)
        else:
            raise ValueError('Invalid normalization strategy')

    def __getitem__(self, idx):

        img, mask = self.images[idx], self.masks[idx]

        if self.transforms is not None:
            img, mask = self.transforms(img, mask)

        return img, mask

    def __len__(self):

        return len(self.images)


def get_mid_planes_selection(mask, q=0.5):
    """
    Returns a list of interger indices with which the central fraction of planes can be selected.
    """
    
    n_planes = mask.shape[0]
    start = int( (0.5 - q/2) * n_planes )
    stop =  int( (0.5 + q/2) * n_planes )

    # clip to not go oob
    start = max(start, 0)
    stop = min(stop, n_planes)
    
    selection = list(range(start, stop))
    return selection


def get_labeled_planes_selection(mask, min_labeled_pixels=1):
    """
    Get a boolean selection for the subset of xy planes in which there are at least min_labeled_pixels with nonzero label.
    The last two dimensions are interpreted as yx, the result will have shape (N_labeled_planes, Y, X).
    """

    # sum binarized mask over last 2 dimensions, get selection of planes with enough labeled pixels
    mask_bin = mask > 0
    selection = (
        mask_bin.sum(axis=tuple(range(mask.ndim - 2, mask.ndim))) >= min_labeled_pixels
    )

    return selection


def sliding_window_planewise_padded(img, window_size=3):
    """
    
    (will take form of channels in Conv.Layer input)
    """    
    padding = ((window_size//2, (window_size-1)//2), ) + ((0,0),) * (img.ndim - 1)
    res = np.pad(img, padding)
    res = np.lib.stride_tricks.sliding_window_view(res, window_size, 0)
    res = res.transpose((0, img.ndim) + tuple(range(1, img.ndim)))
    return res


In [ ]:
from unet import LightningUNet
from lightning import pytorch as L
from torch.nn.functional import cross_entropy, binary_cross_entropy, sigmoid, binary_cross_entropy_with_logits
from torchvision.ops import sigmoid_focal_loss


def get_masked_loss_input(logits_pred, y_gt):

    """
    Get loss function input for sparse segmentation.
    Will select from predicted logits and ground-truth labels 
    """

    # mask == 0 indicates unlabelled
    selection = y_gt > 0

    # select logits -> (npix, c) array 
    logits_selected = torch.transpose(logits_pred, 0, 1)[:, selection].T

    # select nonzero from gt mask, subtract 1 (label==1 in sparse indicates background==0, 2 indicates 1, ...)
    y_selected_corrected = y_gt[selection] - 1
    
    return logits_selected, y_selected_corrected



class SparseSegmentationUNet(LightningUNet):

    """
    Training subclass of ligthning UNet for sparse labels
    """

    def training_step(self, batch, batch_idx):
        
        # apply net
        x, y = batch
        yp = self.forward(x)

        # select pixels labeled in GT, calculate CE for those
        logits_selected, y_selected = get_masked_loss_input(yp, y)
        loss = cross_entropy(logits_selected, y_selected)

        self.log('train_loss', loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)


class DenseSegmentationUNet(LightningUNet):
    """
    Training subclass for conventional, full, labels
    """

    def training_step(self, batch, batch_idx):
        return self._train_val_step(batch, 'loss_train')

    def validation_step(self, batch, batch_idx):
        return self._train_val_step(batch, 'loss_val')

    def _train_val_step(self, batch, log_prefix):
        
        # apply net
        x, y = batch
        yp = self.forward(x)

        # only one output channel, use BCE
        if yp.shape[1] == 1:
            yp = yp[:, 0]
            loss = binary_cross_entropy_with_logits(yp, y.float(), reduction='mean')
        # multiple output channels
        else:
            loss = cross_entropy(yp, y)

        
        self.log(log_prefix, loss, prog_bar=True)

        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)



In [ ]:
base_path = '/data/agl_data/AndreasMaiser/NSD/26AM06-02_2'
image_subfolder = 'patches_gfp+'
label_subfolder = 'patches-segmentation-threshold'

img_files = natsorted(glob(os.path.join(base_path, image_subfolder, "*_ch1*.tif")))
mask_files = natsorted(glob(os.path.join(base_path, label_subfolder, "*.tif")))

# random resize crop (should work even for smaller img) and flips
tr = v2.Compose(
    [
        v2.RandomResizedCrop((128,128), scale=(0.9, 1.0)),
        v2.RandomHorizontalFlip(),
        v2.RandomVerticalFlip()
    ]
)

# plane selector funtions
# NOTE: we first select labelled planes, than the middle of those
selectors = [
    # partial(get_labeled_planes_selection, min_labeled_pixels=20),
    # partial(get_mid_planes_selection, q=0.5),
]

dataset = SparseLabeledImageDataset(img_files, mask_files, transforms=tr,
                                    plane_selectors=selectors,
                                    normalization_strategy=NormalizationStrategy.PER_IMAGE,
                                    plane_sliding_window=5)

# TODO: select a subset of input files to use as a validation set
dataset_val = None

In [ ]:
img, mask = dataset[np.random.randint(0, len(dataset))]

fig, axs = plt.subplots(ncols=2)
axs[0].imshow(img.min(axis=0)[0])
axs[1].imshow(mask.squeeze())


len(dataset)
img.shape

In [ ]:
from torch.utils.data import DataLoader

net = DenseSegmentationUNet(1, [64, 128, 128], input_channels=5)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

trainer = L.Trainer(
    logger=L.loggers.CSVLogger(""),
    log_every_n_steps=np.ceil(len(dataset) / loader.batch_size),
    max_epochs=300,
)

trainer.fit(net, loader)


# x, y = next(iter(loader))
# yp = net(x)

# yp = sigmoid(yp[:, 0])
# binary_cross_entropy(yp, y)

In [ ]:
from lightning.pytorch.utilities.model_summary import ModelSummary

net_inference =  LightningUNet.load_from_checkpoint('lightning_logs/version_34/checkpoints/epoch=19-step=5660.ckpt').eval()
# net_inference =  LightningUNet.load_from_checkpoint('/Users/david/Desktop/jurkat_nucleolin/unet_nucleolus_001/checkpoints/epoch=299-step=3300.ckpt').eval()


ModelSummary(net_inference, max_depth=3)

In [ ]:
test_file = '/Volumes/nn/Julia Vogtmann/Microscopy/26JV_018/tif/0001_ch0.tif'

# add two dummy dimensions (batch size, channels)
img = torch.from_numpy(imread(test_file)).float()[:, torch.newaxis, torch.newaxis]

# ALTERNATIVE with full loader (different batch size, etc.):
# img = torch.from_numpy(imread(test_file)).float()[:, torch.newaxis]
# predict_ds = torch.utils.data.TensorDataset(img)
# predict_loader = torch.utils.data.DataLoader(predict_ds, 1)


trainer = L.Trainer(enable_checkpointing=False, logger=False)
with torch.no_grad():
    pred = trainer.predict(net_inference, img)
    pred = torch.concat(pred)
    probs = torch.softmax(pred, 1)
    pred_labels = pred.argmax(1)

In [ ]:
import nd2

test_file = '/Volumes/agl_data/AndreasMaiser/NSD/26AM06-02_2/0010.nd2'
img = nd2.imread(test_file, dask=True, xarray=True)
img = img.isel(C=1).values
img = SparseLabeledImageDataset._normalize_intensities(img, NormalizationStrategy.PER_IMAGE)

sliding_window = 5
if sliding_window > 1:
    img = sliding_window_planewise_padded(img, sliding_window)
    img = torch.from_numpy(img).float()[:, torch.newaxis, :]
else:
    img = torch.from_numpy(img).float()[:, torch.newaxis, torch.newaxis]


trainer = L.Trainer(enable_checkpointing=False, logger=False)
with torch.no_grad():
    pred = trainer.predict(net_inference, img)
    pred = torch.concat(pred)

    if pred.shape[1] == 1:
        probs = torch.sigmoid(pred[:,0])
        pred_labels = probs > 0.5
    else:
        probs = torch.softmax(pred, 1)
        pred_labels = pred.argmax(1)

In [ ]:
import napari

if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.Viewer()
viewer.add_image(img[:,0,sliding_window//2])

# view predictions of a single class
# viewer.add_labels((pred_labels==2).int())

# ALTERNATIVE: all classes
viewer.add_labels(pred_labels.int())


viewer.add_image(probs)